In [10]:
# === ENV FEDERATED INITIALIZATION ===
import numpy as np
import pandas as pd
from tqdm import tqdm
import hashlib

print("ENV federated aktif — siap menjalankan FedCache & ODE.")


ENV federated aktif — siap menjalankan FedCache & ODE.


In [12]:
# === LOAD ARTIFACTS IN FEDERATED ENV ===

import numpy as np
from tensorflow.keras.models import load_model

X_wave = np.load("X_wave.npy")
X_feat = np.load("X_feat.npy")
logits_ac3 = np.load("logits_ac3.npy")

model = load_model("ac3_teacher_model.h5", compile=False)

print("Artifacts loaded successfully in federated environment.")


Artifacts loaded successfully in federated environment.


In [13]:
# === STEP 5A: EKSTRAK EMBEDDING UNTUK FEDCACHE (ENV FEDERATED) ===

from tensorflow.keras.models import Model

# extractor mengambil output BiLSTM sebelum gabung fitur
extractor = Model(
    inputs=model.inputs,
    outputs=model.get_layer(index=6).output
)

embeddings = []

for i in tqdm(range(len(X_wave)), desc="Extracting embeddings for FedCache"):
    xw = X_wave[i:i+1]
    xf = X_feat[i:i+1]
    emb = extractor.predict({"waveform": xw, "features": xf}, verbose=0)
    embeddings.append(emb[0])

embeddings = np.array(embeddings)
embeddings.shape


Extracting embeddings for FedCache:   0%|          | 0/2515 [00:00<?, ?it/s]

Extracting embeddings for FedCache: 100%|██████████| 2515/2515 [00:49<00:00, 50.55it/s]


(2515, 300, 64)

In [15]:
# === STEP 5B: HASHING EMBEDDING UNTUK FEDCACHE (ENV FEDERATED) ===

import hashlib

def hash_embedding(vec):
    vec_bytes = vec.tobytes()
    return hashlib.sha1(vec_bytes).hexdigest()[:16]

hashes = []

for e in tqdm(embeddings, desc="Hashing embeddings"):
    hashes.append(hash_embedding(e))

len(hashes)


Hashing embeddings: 100%|██████████| 2515/2515 [00:00<00:00, 18987.60it/s]


2515

In [16]:
# === STEP 5C: MEMBANGUN FEDCACHE (HASH → LOGIT TEACHER) ===

cache = {}

for h, logit in zip(hashes, logits_ac3):
    cache[h] = float(logit)   # pastikan float untuk JSON-friendly

print("FedCache size:", len(cache))


FedCache size: 2515


In [17]:
# === STEP 5D: OFFLOADING DECISION ENGINE (ODE) ===

def ODE_decision(embedding, cache_dict):
    h = hash_embedding(embedding)
    if h in cache_dict:
        return "CACHE_HIT", cache_dict[h]
    else:
        return "CACHE_MISS", None

# contoh uji
status, value = ODE_decision(embeddings[0], cache)
print("ODE:", status, "Value:", value)


ODE: CACHE_HIT Value: 3.768470048904419


In [18]:
# === STEP 5E: SIMPAN FEDCACHE UNTUK SBC (CSV) ===

import pandas as pd

fedcache_df = pd.DataFrame({
    "hash": hashes,
    "logit": logits_ac3
})

fedcache_df.to_csv("fedcache_data.csv", index=False)
print("FedCache saved as fedcache_data.csv")


FedCache saved as fedcache_data.csv


In [19]:
# === OPTIONAL: SIMPAN FEDCACHE DALAM JSON ===

import json

with open("fedcache_data.json", "w") as f:
    json.dump(cache, f)

print("FedCache saved as fedcache_data.json")


FedCache saved as fedcache_data.json


In [20]:
np.save("embeddings.npy", embeddings)
print("embeddings.npy saved.")


embeddings.npy saved.


In [ ]:
# sampai tahap ini kembali ke env seismic untuk melanjutkan SIMULASI